In [2]:
import os
import glob
import cv2
import numpy as np
import matplotlib.pyplot as plt # visualizeメソッドで必要
import concurrent.futures
from tqdm import tqdm # 進捗表示のために追加（推奨）

# ===================================================================
# 1. 単一画像の処理クラス（変更なし）
# ===================================================================
class Size_taikakusen:
    def __init__(self, mask_path):
        self.mask_path = mask_path
        self.mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if self.mask is None:
            raise ValueError(f"画像が読み込めません: {mask_path}")
        _, self.mask_bin = cv2.threshold(self.mask, 127, 255, cv2.THRESH_BINARY)
        self.center_of_mass = self._compute_center_of_mass()

    def _compute_center_of_mass(self):
        contours, _ = cv2.findContours(self.mask_bin, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            # print("輪郭が見つかりません")
            return np.array([0, 0])
        largest_contour = max(contours, key=cv2.contourArea)
        M = cv2.moments(largest_contour)
        if M["m00"] == 0:
            # print("面積ゼロの輪郭")
            return np.array([0, 0])
        cx = int(M["m10"] / M["m00"])
        cy = int(M["m01"] / M["m00"])
        return np.array([cx, cy])

    def compute_diameters(self):
        min_distance = float('inf')
        max_distance = 0
        min_points = None
        max_points = None

        for angle in range(0, 180):
            angle_rad = np.deg2rad(angle)
            dx = np.cos(angle_rad)
            dy = np.sin(angle_rad)

            # 正方向の探索
            x, y = self.center_of_mass.copy()
            farthest_point = None
            while 0 <= int(x) < self.mask.shape[1] and 0 <= int(y) < self.mask.shape[0]:
                if self.mask_bin[int(y), int(x)] == 0: # 輪郭の外に出た瞬間
                    farthest_point = (x, y)
                    break
                x += dx
                y += dy
            
            # 負方向の探索
            x, y = self.center_of_mass.copy()
            nearest_point = None
            while 0 <= int(x) < self.mask.shape[1] and 0 <= int(y) < self.mask.shape[0]:
                if self.mask_bin[int(y), int(x)] == 0: # 輪郭の外に出た瞬間
                    nearest_point = (x, y)
                    break
                x -= dx
                y -= dy

            if farthest_point is not None and nearest_point is not None:
                dist = np.linalg.norm(np.array(farthest_point) - np.array(nearest_point))
                if dist < min_distance:
                    min_distance = dist
                    min_points = (farthest_point, nearest_point)
                if dist > max_distance:
                    max_distance = dist
                    max_points = (farthest_point, nearest_point)

        self.min_distance = min_distance if min_distance != float('inf') else 0
        self.max_distance = max_distance
        self.min_points = min_points
        self.max_points = max_points
        return (self.min_distance + self.max_distance) / 2 # 平均を返す
    
    # visualize メソッドは変更なし

# ===================================================================
# 2. 各プロセスで実行されるヘルパー関数
# ===================================================================
def process_single_image(file_path):
    """
    単一の画像ファイルを処理し、ファイル名と計算結果のタプルを返す。
    エラーが発生した場合は None を返す。
    """
    try:
        file_name = os.path.basename(file_path)
        size_calc = Size_taikakusen(file_path)
        avg_diameter = size_calc.compute_diameters()
        return (file_name, avg_diameter)
    except Exception as e:
        print(f"⚠️ エラー発生 ({os.path.basename(file_path)}): {e}")
        return None

# ===================================================================
# 3. フォルダ単位の処理クラス（マルチプロセス対応に修正）
# ===================================================================
class Size_folder_taikakusen:
    def __init__(self, base_dir, mask_dir="mask", subfolder="collage_1"):
        self.base_dir = base_dir
        self.mask_dir = mask_dir
        self.subfolder = subfolder
        self.input_folder = os.path.join(base_dir, mask_dir, subfolder)
        self.data_tapple = []
        print("探索対象フォルダ:", self.input_folder)

    def run(self):
        if not os.path.isdir(self.input_folder):
            print(f"⚠️ フォルダが存在しません: {self.input_folder}")
            return []

        # 処理対象の画像ファイルパスのリストを作成
        image_paths = [
            os.path.join(self.input_folder, f)
            for f in os.listdir(self.input_folder)
            if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))
        ]

        if not image_paths:
            print(f"⚠️ フォルダ内に処理対象の画像がありません: {self.input_folder}")
            return []
        
        print(f"📂 {len(image_paths)} 枚の画像をマルチプロセスで処理します。")

        # ProcessPoolExecutorを使用して並列処理を実行
        with concurrent.futures.ProcessPoolExecutor() as executor:
            # executor.mapを使い、各画像パスに対してprocess_single_imageを並列実行
            results_iterator = executor.map(process_single_image, image_paths)

            # tqdmを使って進捗バーを表示しつつ、Noneでない結果のみをリストに格納
            self.data_tapple = []
            desc = f"処理中 ({self.subfolder})"
            for result in tqdm(results_iterator, total=len(image_paths), desc=desc):
                if result is not None:
                    self.data_tapple.append(result)
        
        return self.data_tapple

# ===================================================================
# 4. 実行例
# ===================================================================


In [3]:
# マルチプロセス対応版コードの実行部分を以下のように修正

if __name__ == '__main__':
    import time # timeモジュールをインポート

    # --- ユーザー設定 ---
    data_variable = "0827_seido"
    base_dir_path = f"/home/data/{data_variable}"
    subfolder_name = "A"

    print("--- マルチプロセス版の計測を開始 ---")
    # --- 時間計測開始 ---
    start_time = time.perf_counter()

    # --- 処理の実行 ---
    size_analyzer = Size_folder_taikakusen(base_dir=base_dir_path, subfolder=subfolder_name)
    results = size_analyzer.run()
    
    # --- 時間計測終了 ---
    end_time = time.perf_counter()
    elapsed_time = end_time - start_time

    print(f"\n[結果] 処理された画像数: {len(results)}")
    print(f"🚀 マルチプロセス版の実行時間: {elapsed_time:.2f} 秒")

--- マルチプロセス版の計測を開始 ---
探索対象フォルダ: /home/data/0827_seido/mask/A
📂 496 枚の画像をマルチプロセスで処理します。


処理中 (A): 100%|██████████| 496/496 [00:10<00:00, 48.68it/s]


[結果] 処理された画像数: 496
🚀 マルチプロセス版の実行時間: 10.31 秒


↓マルチプロセス前，比較用

In [8]:
print(results)

[('IMG_66639_mask.png', 414.9999999999993), ('IMG_66517_mask.png', 395.00000000000125), ('IMG_66627_mask.png', 409.4999999999998), ('IMG_6664_mask.png', 433.500000000001), ('IMG_64219_mask.png', 434.0000000000001), ('IMG_64544_mask.png', 388.0000000000001), ('IMG_66634_mask.png', 398.99999999999994), ('IMG_6445_mask.png', 412.50000000000057), ('IMG_64621_mask.png', 489.9999999999985), ('IMG_66531_mask.png', 439.0000000000015), ('IMG_66236_mask.png', 392.0000000000019), ('IMG_66711_mask.png', 452.50000000000193), ('IMG_6625_mask.png', 420.5000000000016), ('IMG_64251_mask.png', 393.99999999999994), ('IMG_64410_mask.png', 459.50000000000296), ('IMG_6623_mask.png', 442.00000000000153), ('IMG_64350_mask.png', 408.5000000000002), ('IMG_66727_mask.png', 435.9999999999985), ('IMG_66515_mask.png', 418.99999999999875), ('IMG_64437_mask.png', 424.5000000000024), ('IMG_6621_mask.png', 428.00000000000205), ('IMG_64619_mask.png', 465.49999999999955), ('IMG_6456_mask.png', 420.0000000000038), ('IMG_6

In [9]:
import os
import cv2
import numpy as np

class Size_folder:
    def __init__(self, base_dir, mask_dir="mask", subfolder="collage_1"):
        self.base_dir = base_dir
        self.mask_dir = mask_dir
        self.subfolder = subfolder

        # マスク画像があるフォルダのパスを組み立てる
        self.input_folder = os.path.join(base_dir, mask_dir, subfolder)
        self.data_tapple = []

        print("探索対象フォルダ:", self.input_folder)

    def run(self):
        if not os.path.isdir(self.input_folder):
            print(f"⚠️ フォルダが存在しません: {self.input_folder}")
            return self.data_tapple

        for file in os.listdir(self.input_folder):
            if file.endswith(('.png', '.jpg', '.jpeg', '.bmp', '.JPEG')):
                file_path = os.path.join(self.input_folder, file)
                image = cv2.imread(file_path, cv2.IMREAD_GRAYSCALE)

                if image is None:
                    print(f"⚠️ 画像読み込み失敗: {file_path}")
                    continue

                # 二値化
                _, binary = cv2.threshold(image, 128, 255, cv2.THRESH_BINARY)

                # 白ピクセルカウント
                white_pixel_count = np.sum(binary == 255)

                # 結果保存
                self.data_tapple.append((file, white_pixel_count))

        return self.data_tapple

class Size_file:
    def __init__(self, image_path):
        self.image_path = image_path

    def count_white_pixels(self):
        image = cv2.imread(self.image_path, cv2.IMREAD_GRAYSCALE)

        if image is None:
            raise FileNotFoundError(f"画像が読み込めませんでした: {self.image_path}")

        # 二値化
        _, binary = cv2.threshold(image, 128, 255, cv2.THRESH_BINARY)

        # 白ピクセルのカウント
        white_pixel_count = np.sum(binary == 255)

        return white_pixel_count
    
# size1 = Size_file('/home/data/maesyori_img/mask/collage_1/mask_1_1.jpg')
# white_count = size1.count_white_pixels()
# print(white_count)

# size2 = Size_folder(['/home/data/maesyori_img/mask/collage_1'])
# white_count_list = size2.count_white_pixels()
# print(white_count_list)

import cv2
import numpy as np
import matplotlib.pyplot as plt

class Size_taikakusen:
    def __init__(self, mask_path):
        self.mask_path = mask_path
        self.mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if self.mask is None:
            raise ValueError(f"画像が読み込めません: {mask_path}")
        _, self.mask_bin = cv2.threshold(self.mask, 127, 255, cv2.THRESH_BINARY)
        self.center_of_mass = self._compute_center_of_mass()

    def _compute_center_of_mass(self):
        contours, _ = cv2.findContours(self.mask_bin, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            print("輪郭が見つかりません")
            return np.array([0, 0])
        largest_contour = max(contours, key=cv2.contourArea)
        M = cv2.moments(largest_contour)
        if M["m00"] == 0:
            print("面積ゼロの輪郭")
            return np.array([0, 0])
        cx = int(M["m10"] / M["m00"])
        cy = int(M["m01"] / M["m00"])
        return np.array([cx, cy])

    def compute_diameters(self):
        min_distance = float('inf')
        max_distance = 0
        min_points = None
        max_points = None

        for angle in range(0, 180):
            angle_rad = np.deg2rad(angle)
            dx = np.cos(angle_rad)
            dy = np.sin(angle_rad)

            x, y = self.center_of_mass
            farthest_x, farthest_y = None, None
            while 0 <= int(x) < self.mask.shape[1] and 0 <= int(y) < self.mask.shape[0]:
                if self.mask_bin[int(y), int(x)] == 0:
                    farthest_x, farthest_y = x, y
                    break
                x += dx
                y += dy

            x, y = self.center_of_mass
            nearest_x, nearest_y = None, None
            while 0 <= int(x) < self.mask.shape[1] and 0 <= int(y) < self.mask.shape[0]:
                if self.mask_bin[int(y), int(x)] == 0:
                    nearest_x, nearest_y = x, y
                    break
                x -= dx
                y -= dy

            if farthest_x is not None and nearest_x is not None:
                dist = np.sqrt(((farthest_x - nearest_x) - 2)**2 + ((farthest_y - nearest_y) - 2)**2)
                if dist < min_distance:
                    min_distance = dist
                    min_points = ((farthest_x, farthest_y), (nearest_x, nearest_y))
                if dist > max_distance:
                    max_distance = dist
                    max_points = ((farthest_x, farthest_y), (nearest_x, nearest_y))

        self.min_distance = min_distance
        self.max_distance = max_distance
        self.min_points = min_points
        self.max_points = max_points
        return (min_distance + max_distance) / 2  # 平均を返す

    def visualize(self):
        image = cv2.cvtColor(self.mask, cv2.COLOR_GRAY2BGR)
        cx, cy = self.center_of_mass
        cv2.circle(image, (cx, cy), 5, (255, 0, 0), -1)

        if self.min_points:
            cv2.line(image,
                     (int(self.min_points[0][0]), int(self.min_points[0][1])),
                     (int(self.min_points[1][0]), int(self.min_points[1][1])),
                     (0, 255, 255), 2)

        if self.max_points:
            cv2.line(image,
                     (int(self.max_points[0][0]), int(self.max_points[0][1])),
                     (int(self.max_points[1][0]), int(self.max_points[1][1])),
                     (0, 0, 255), 2)

        plt.figure(figsize=(8, 6))
        plt.imshow(image[..., ::-1])
        plt.title("Mask with Min (Yellow) and Max (Red) Diameters")
        plt.axis("off")
        plt.tight_layout()
        plt.show()
    
    
class Size_folder_taikakusen:
    def __init__(self, base_dir, mask_dir="mask", subfolder="collage_1"):
        self.base_dir = base_dir
        self.mask_dir = mask_dir
        self.subfolder = subfolder

        self.input_folder = os.path.join(base_dir, mask_dir, subfolder)
        self.data_tapple = []

        print("探索対象フォルダ:", self.input_folder)

    def run(self):
        if not os.path.isdir(self.input_folder):
            print(f"⚠️ フォルダが存在しません: {self.input_folder}")
            return self.data_tapple

        for file in os.listdir(self.input_folder):
            if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                file_path = os.path.join(self.input_folder, file)

                try:
                    size_calc = Size_taikakusen(file_path)
                    avg_diameter = size_calc.compute_diameters()
                    self.data_tapple.append((file, avg_diameter))
                except Exception as e:
                    print(f"⚠️ エラー（{file}）: {e}")

        return self.data_tapple

In [10]:
data = "0827_seido"
start_time = time.time()
size_tappleA = Size_folder_taikakusen(base_dir=f"/home/data/{data}",subfolder="A")
result_sizeA = size_tappleA.run()
end_time = time.time()
print(f"処理時間: {end_time - start_time:.2f} 秒")

探索対象フォルダ: /home/data/0827_seido/mask/A
処理時間: 67.06 秒


In [7]:
print(result_sizeA)

[('IMG_66639_mask.png', 414.0287980147281), ('IMG_66517_mask.png', 394.09677527721), ('IMG_66627_mask.png', 407.3002822169676), ('IMG_6664_mask.png', 431.64737005919847), ('IMG_64219_mask.png', 432.3206530905277), ('IMG_64544_mask.png', 386.89205266520537), ('IMG_66634_mask.png', 398.00535574145727), ('IMG_6445_mask.png', 411.43025917798064), ('IMG_64621_mask.png', 487.43844637788663), ('IMG_66531_mask.png', 437.6525927942614), ('IMG_66236_mask.png', 392.39187854119), ('IMG_66711_mask.png', 450.6390412004932), ('IMG_6625_mask.png', 419.0051525925226), ('IMG_64251_mask.png', 392.6794469254845), ('IMG_64410_mask.png', 457.3637420374391), ('IMG_6623_mask.png', 441.87975593781727), ('IMG_64350_mask.png', 406.46354633029387), ('IMG_66727_mask.png', 434.4914270422248), ('IMG_66515_mask.png', 417.1704991025266), ('IMG_64437_mask.png', 424.20810925915345), ('IMG_6621_mask.png', 426.7435516506119), ('IMG_64619_mask.png', 464.64471379331485), ('IMG_6456_mask.png', 417.52119868125493), ('IMG_6646